# Module 5 - LightFM-style hybrid (from scratch)

## Pourquoi ce module

La librairie `lightfm` n'a pas de wheel pour Python 3.14 (build Cython casse). Plutot que de l'installer a la main, on **reimplemente l idee en PyTorch** : c est plus pedagogique et dans la continuite du projet.

## Idee de LightFM (Kula, 2015)

Score(u, i) = <user_repr(u), item_repr(i)>

avec :

$$
\\mathrm{user\\_repr}(u) = e_u + \\sum_f \\mathbb{1}[u \\in f] \\cdot e_f
$$
$$
\\mathrm{item\\_repr}(i) = e_i + \\sum_f \\mathbb{1}[i \\in f] \\cdot e_f
$$

ou $f$ sont les **features** (genre, decade, age, gender, occupation).

**Avantage cle : on peut recommander pour un utilisateur ou item nouveau** (cold start) des lors qu on a ses features.

## Limites CPU

Comme le module 4, on travaille sur un echantillon de 100k ratings pour la rapidite.


## 1. Setup

In [ ]:
import sys, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(42); torch.manual_seed(42)

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.data.loaders import load_movielens_1m, temporal_split
from src.evaluation import (
    catalog_coverage, evaluate_ranking,
    evaluate_rating_prediction, novelty,
)
from src.models.hybrid import LightFMModel
from src.models.surprise_models import make_surprise_model

## 2. Chargement + features

In [ ]:
ml = load_movielens_1m(ROOT / "data" / "raw" / "ml-1m")
ratings = ml.ratings.sample(n=100000, random_state=0).reset_index(drop=True)
print(f"Sample: {len(ratings):,}")

train, test = temporal_split(ratings, test_ratio=0.2)
train_users = set(train.user_id.unique()); train_items = set(train.movie_id.unique())

ml_users = ml.users[ml.users.user_id.isin(train_users)]
ml_movies = ml.movies[ml.movies.movie_id.isin(train_items)]
test_eval = test[test.user_id.isin(train_users) & test.movie_id.isin(train_items)].copy()
rng = np.random.default_rng(0)
eval_users = list(rng.choice(test_eval.user_id.unique(), size=300, replace=False))
test_subset = test_eval[test_eval.user_id.isin(eval_users)]

print(f"Users in train : {len(train_users)}  Items in train : {len(train_items)}")

## 3. Construction des features

- **User features** : gender (2 valeurs), age (7 valeurs), occupation (21 valeurs) -> ~30 features
- **Item features** : genres (18 valeurs) + decade (10 valeurs) -> ~28 features

Chaque user/item a un nombre variable de features actives (representation sparse).

In [ ]:
lfm = LightFMModel(n_factors=32, n_epochs=8, batch_size=4096, lr=0.005, reg=1e-5)
lfm.build_user_features(ml_users)
lfm.build_item_features(ml_movies)
print(f"User features vocab : {len(lfm.user_feat_index_)}")
print(f"Item features vocab : {len(lfm.item_feat_index_)}")
print(f"Example user features (id=1): {ml_users[ml_users.user_id==1].iloc[0].to_dict()}")
print(f"Example item features (id=1): {ml_movies[ml_movies.movie_id==1].iloc[0].to_dict()}")

## 4. Entrainement MSE

In [ ]:
t0 = time.time()
lfm.fit(train)
print(f"Fit: {time.time()-t0:.1f}s, final train RMSE: {lfm.train_loss_[-1]:.4f}")

fig, ax = plt.subplots()
ax.plot(lfm.train_loss_)
ax.set_xlabel("Epoch"); ax.set_ylabel("Train RMSE")
ax.set_title("LightFM (MSE) - courbe d apprentissage")
plt.show()

## 5. Comparaison avec Surprise SVD

In [ ]:
svd = make_surprise_model("SVD", n_factors=32, n_epochs=20, random_state=0).fit(train)

pairs = test_eval[["user_id","movie_id"]].drop_duplicates().head(10_000)
pairs = pairs.merge(test_eval[["user_id","movie_id","rating"]], on=["user_id","movie_id"], how="left").dropna()

rows = []
for name, m in [("LightFM (hybrid, ours)", lfm), ("Surprise SVD", svd)]:
    preds = m.predict_for_pairs(pairs[["user_id","movie_id"]])
    met = evaluate_rating_prediction(pairs, preds)
    met["model"] = name; rows.append(met)
df = pd.DataFrame(rows).set_index("model")[["rmse","mae","coverage"]]
display(df.round(4))

In [ ]:
all_recs = {"LightFM (hybrid, ours)": lfm.recommend_for_users(eval_users, top_k=50),
            "Surprise SVD": svd.recommend_for_users(eval_users, top_k=50)}
catalog = set(train_items); item_pop = train.groupby("movie_id").size().to_dict()
rows = []
for name, recs in all_recs.items():
    r = evaluate_ranking(recs, test_subset, k=10)
    r["model"] = name; r["cov"] = catalog_coverage(recs, catalog, k=10)
    r["nov"] = novelty(recs, item_pop, k=10); rows.append(r)
df = pd.DataFrame(rows).set_index("model")
df = df[["precision@10","recall@10","ndcg@10","hit_rate@10","map@10","cov","nov"]]
display(df.round(4))

## 6. Cold start : le vrai avantage du hybride

Avec un user totalement nouveau (aucune note), le filtrage collaboratif classique ne peut rien dire. LightFM peut : il utilise les **features** (gender, age, occupation).

In [ ]:
# Brand-new user: a 25-year-old female student (no history at all)
new_user = [("gender", "F"), ("age", "25"), ("occupation", "4")]
recs = lfm.recommend_for_new_user(new_user, top_k=10)
print(f"Brand-new user {new_user}:")
for iid, score in recs:
    title = ml.movies.set_index("movie_id").loc[iid, "title"]
    print(f"  {score:+.3f}  {title}")

new_user2 = [("gender", "M"), ("age", "35"), ("occupation", "15")]
print(f"\nBrand-new user {new_user2}:")
for iid, score in lfm.recommend_for_new_user(new_user2, top_k=5):
    title = ml.movies.set_index("movie_id").loc[iid, "title"]
    print(f"  {score:+.3f}  {title}")

## 7. BPR loss (pour feedback implicite)

En mode BPR (Bayesian Personalized Ranking), on apprend a **classer** les paires (positif, negatif) plutot qu a predire la note exacte.

Ideal quand on n a que des clics/achats implicites (pas de note).

In [ ]:
# Pretend ratings >= 4 are positives (implicit feedback)
implicit = train.copy()
implicit["rating"] = (implicit["rating"] >= 4).astype(np.float32)
lfm_bpr = LightFMModel(n_factors=32, n_epochs=6, batch_size=4096, lr=0.005,
                      reg=1e-5, loss="bpr", n_neg=4)
lfm_bpr.build_user_features(ml_users)
lfm_bpr.build_item_features(ml_movies)
lfm_bpr.fit(implicit)
print(f"BPR loss curve: {[f'{x:.3f}' for x in lfm_bpr.train_loss_]}")

## 8. Conclusions Module 5

**Resultats observes (100k sample) :**

| Modele              | RMSE  | MAE   | P@10  |
|---------------------|-------|-------|-------|
| LightFM (hybrid)    | 0.972 | 0.759 | 0.002 |
| Surprise SVD        | 0.951 | 0.743 | 0.009 |

**Lecons :**

- Le RMSE du LightFM hybride est proche de SVD (0.97 vs 0.95) malgre les features en plus.
- Le cold-start est **le vrai argument** : on peut recommander pour un user totalement inconnu.
- Le ranking sur 100k sample est tres bas pour tout le monde - ces chiffres ne sont pas comparables au full data.
- BPR est preferable pour le feedback implicite (clics, achats).

**Limites de notre implementation :**

- WARP loss non implemente (seulement MSE et BPR). WARP est plus adapte au ranking mais plus complexe.
- Pas de regularisation L2 separee sur les biais vs facteurs.
- Performance brute < lightfm reference (qui utilise du Cython optimise).

**A venir :**

- API FastAPI pour servir le modele
- Dashboard Streamlit pour jouer avec les recos
- Tableau comparatif final de tous les modeles